# Figures - the results as pictures

The figures of the README, made from what the scoring saved - the decoder heads of seed 0 and the settings of every run. Nothing is trained or tuned here. Four pictures

- a prediction GIF per radar dataset (glacial lakes, snow, forest types, alpine glaciers). It steps through the Sentinel-1 radar, the truth and the predictions of the frozen satellite DINOv3, student F and student C (without the radar teacher), each with its score on that chip, and ends on all of them together
- a feature-map GIF, what each of the three models sees on one chip of every dataset, the patch features as colours (one PCA per image)
- the results chart, the decoder scores of the validation splits with the seed spread
- the architecture picture, the inputs, the student and the four teacher slots

## How the chips are picked

By a rule on the labels alone, never by how well a model does on them. Two validation chips per dataset with at least 90 percent of the pixels labelled (90 percent of the best chip's share where every chip carries padding, as the alpine tiles do) and the most even class mix (the highest entropy of the class shares), the second one from another region when there is one.

## Runtime and time

The L4 GPU, standard RAM is enough. About 10 minutes, an estimate - the validation splits of the four datasets, the three encoders on 8 chips, the GIFs.

## Data

The validation splits of the four radar datasets, from their private Hugging Face copies when DATASET_HF_OWNER is set, else from their copies on Drive. The decoder heads and run settings of the frozen satellite DINOv3 come from part two's results (REFERENCE_DRIVE_DIR), those of the students from this project's results, the students from the training run.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/figures/ - predictions_glacial_lakes.gif, predictions_snow.gif, predictions_forty_v1.gif, predictions_glavitu_alp.gif, features.gif, results.png, architecture.png and chips.json with the picked chips. A figure on Drive is made again on every run, the rule picks the same chips.

## How to run

Run the cells from top to bottom. The last cell flushes Drive and then releases the Colab machine (set END_RUNTIME = False in that cell to keep the runtime), and a cell that fails stops the run before it.


In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. Tokens, model cache, GPU check, folders, the validation splits on the runtime disk, the saved runs.
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sartransfer import env, figures
from sartransfer.distil import score, stage0, train
from sartransfer.domains import hub, prepared
from sartransfer.domains import run as run_step
from sartransfer.domains.prepared import load_manifest
from sartransfer.models import LAST

start = time.perf_counter()
assert torch.cuda.is_available(), "no GPU, pick the L4 runtime and connect again"
DEVICE = "cuda"
MODEL_CACHE = env.model_cache_on_drive()       # before transformers or huggingface_hub load
assert MODEL_CACHE is not None, "Google Drive is not mounted, run the mount cell first"
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the gated DINOv3 weights and the private data copies")
assert HF_TOKEN, "no Hugging Face token was given"
DATA_TOKEN = os.environ.get("HF_WRITE_TOKEN") or HF_TOKEN   # the private dataset copies, as notebook 03
DATASET_OWNER = os.environ.get("DATASET_HF_OWNER", "").strip()   # optional, without it the Drive copies are used
REFERENCE_DIR = env.require("REFERENCE_DRIVE_DIR", "the folder under MyDrive with part two's results")

DATASETS = ("glacial_lakes", "snow", "forty_v1", "glavitu_alp")
SPLIT = "val"
N_CHIPS = 2                                      # chips per dataset in a prediction GIF
STUDENTS = {"student-f": "student_F", "student-c": "student_C"}   # encoder name and its folder of notebook 02
# the models of the figures - name in the figures, encoder, layer, input mode, the folder of its saved runs
RESULTS_FOLDER = "sar-transfer-distillation"     # the Drive folder this notebook writes to, a new name re-runs it from scratch
CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
PREP = CONTENT / "sar-transfer-distillation" / "prepared"
RES = MYDRIVE / "sar-transfer-distillation" / "results"
REF = MYDRIVE / REFERENCE_DIR / "results"
RUNS = MYDRIVE / "sar-transfer-distillation" / f"runs_{train.TrainConfig().steps}"
OUT = MYDRIVE / RESULTS_FOLDER / "figures"            # the figures, the runs above are read only
MODELS = {"Frozen DINOv3 SAT": ("dinov3-l-sat", run_step.PRE_REGISTERED_LAYERS["dinov3-l-sat"], "vv_vh_diff", REF),
          "Student F": ("student-f", LAST, "co_cross", RES),
          "Student C (no radar teacher)": ("student-c", LAST, "co_cross", RES)}
CHECKPOINTS = {name: RUNS / folder / train.FINAL for name, folder in STUDENTS.items()}
TIMES = {}


def took(since, name):
    """The time of a cell as text, recorded under `name`."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def run_files(model, dataset):
    """The saved decoder head of seed 0 and the run settings of one model on one dataset."""
    encoder, layer, mode, folder = MODELS[model]
    run = score.run_name(dataset, encoder, layer, mode)
    return folder / f"{run}__decoder__s0.pt", folder / f"{run}__info.json"


for name, path in CHECKPOINTS.items():
    assert path.is_file(), f"{path} is not on Drive, run notebook 02 first"
for model in MODELS:
    for dataset in DATASETS:
        for path in run_files(model, dataset):
            assert path.is_file(), f"{path} is not on Drive, run notebook 03 first (or part two's scoring)"
for name in DATASETS:
    files = prepared.split_files((SPLIT,))
    if (PREP / name / "manifest.json").is_file() and all((PREP / name / f).is_file() for f in files):
        print(f"{name}  on the runtime disk already")
        continue
    source = None
    if DATASET_OWNER:
        try:                                       # first the private Hugging Face copy, the validation split
            hub.fetch(name, DATASET_OWNER, PREP, token=DATA_TOKEN, splits=(SPLIT,))
            source = "Hugging Face"
        except Exception as err:
            print(f"{name}  not on Hugging Face ({type(err).__name__}, {hide(err)[:200]})")
    if source is None:                             # then the copy on Drive
        copy = MYDRIVE / "datasets" / hub.repo_name(name)
        assert (copy / "manifest.json").is_file(), f"{name} is neither on Hugging Face nor on Drive under {copy}"
        prepared.copy_prepared(copy, PREP, name, splits=(SPLIT,))
        source = "Drive"
    print(f"{name}  from {source}")
print(f"GPU     {torch.cuda.get_device_name(0)}")
print(f"figures go to   {OUT}")
OUT.mkdir(parents=True, exist_ok=True)
print(took(start, "setup"))


In [ ]:
# Chips. Two validation chips per dataset by the rule on the labels alone (figures.pick_chips) - at least 90
# percent labelled (90 percent of the best chip's share for padded tiles), the most even class mix, the second
# from another region. Printed with their class shares.
start = time.perf_counter()
DATA, PICKED = {}, {}
for name in DATASETS:
    man = load_manifest(PREP, name)
    split = prepared.load_split(PREP, name, SPLIT)
    k = len(man["classes"])
    idx = figures.pick_chips(split["labels"], split["meta"]["region"].to_numpy(), k, n=N_CHIPS)
    assert len(idx) == N_CHIPS, f"{name} has fewer than {N_CHIPS} chips that pass the labelled share"
    names = [man["classes"][str(c)] for c in range(k)]
    DATA[name] = {"images": np.asarray(split["images"][idx]), "labels": np.asarray(split["labels"][idx]),
                  "regions": split["meta"]["region"].to_numpy()[idx].astype(str), "classes": names, "k": k}
    PICKED[name] = [int(i) for i in idx]
    for i, j in enumerate(idx):
        shares, labelled = figures.class_shares(split["labels"][j], k)
        mix = ", ".join(f"{names[c]} {shares[c]:.2f}" for c in range(k) if shares[c] > 0)
        print(f"{name:<14} chip {j}, region {DATA[name]['regions'][i]}, labelled {labelled:.2f} - {mix}")
(OUT / "chips.json").write_text(json.dumps(PICKED, indent=1), encoding="utf-8")
print(took(start, "chips"))


In [ ]:
# Predictions. Each model is loaded once and reads the picked chips as the scoring did - the run's own channel
# statistics (its __info.json), the same input mode and layer, its saved decoder head of seed 0, bfloat16 on the
# GPU. The score of every chip is printed. The patch features are kept for the feature-map GIF.
start = time.perf_counter()
from sartransfer.domains.s1input import skip_image
from sartransfer.models.decoder import GridDecoder
from sartransfer.models.encoders import ENCODERS, FrozenEncoder

for name, path in CHECKPOINTS.items():
    score.register_student(name, path)
PRED, FEATURES = {}, {}
for model, (encoder, layer, mode, folder) in MODELS.items():
    enc = FrozenEncoder(ENCODERS[encoder], token=HF_TOKEN, device=DEVICE)
    for name in DATASETS:
        head_file, info_file = run_files(model, name)
        info = json.loads(info_file.read_text(encoding="utf-8"))
        stats = info["stats"]
        in_stats = getattr(enc, "input_stats", {}).get(mode, stats)    # as the scoring chose it
        images = DATA[name]["images"]
        feats = run_step._encode_split(enc, images, in_stats, mode, layer, 4)
        skip = np.round(skip_image(images, stats)[:, 0] * 255).astype(np.uint8)
        head = GridDecoder(feats.shape[-1], n_classes=DATA[name]["k"])
        head.load_state_dict(torch.load(head_file, map_location="cpu", weights_only=True))
        pred = figures.predict(head, feats, skip, device=DEVICE)
        scores = [figures.chip_score(pred[i], DATA[name]["labels"][i], DATA[name]["k"]) for i in range(len(pred))]
        PRED[(model, name)] = (pred, scores)
        FEATURES[(model, name)] = stage0.pca_rgb_per_image(feats)
        print(f"{model:<30} {name:<14} {figures.SCORE_NAMES[name]} per chip {', '.join(f'{s:.2f}' for s in scores)}")
    del enc
    torch.cuda.empty_cache()
print(took(start, "predictions"))


In [ ]:
# GIFs. One prediction GIF per dataset, the chips side by side, then the feature-map GIF with the first chip of
# every dataset, the features shown at the chip's size (each patch one block of colour).
start = time.perf_counter()
for name in DATASETS:
    palette = figures.PALETTES[name]
    chips = []
    for i in range(N_CHIPS):
        chips.append({"radar": figures.radar_rgb(DATA[name]["images"][i]),
                      "truth": figures.label_rgb(DATA[name]["labels"][i], palette),
                      "pred": {m: figures.label_rgb(PRED[(m, name)][0][i], palette) for m in MODELS},
                      "score": {m: PRED[(m, name)][1][i] for m in MODELS},
                      "label": f"region {DATA[name]['regions'][i]}"})
    frames, seconds = figures.prediction_frames(figures.TITLES[name], chips, list(MODELS), palette,
                                                DATA[name]["classes"], figures.SCORE_NAMES[name])
    path = figures.save_gif(frames, OUT / f"predictions_{name}.gif", seconds)
    print(f"{path.name:<32} {len(frames)} frames, {path.stat().st_size / 1e6:.1f} MB")
rows = []
for name in DATASETS:
    size = DATA[name]["labels"].shape[-1]
    feats = {}
    for m in MODELS:
        f = FEATURES[(m, name)][0]
        feats[m] = np.kron(f, np.ones((size // f.shape[0], size // f.shape[1], 1), np.uint8))
    rows.append({"title": figures.TITLES[name], "radar": figures.radar_rgb(DATA[name]["images"][0]),
                 "truth": figures.label_rgb(DATA[name]["labels"][0], figures.PALETTES[name]), "features": feats})
frames, seconds = figures.feature_frames(rows, list(MODELS))
path = figures.save_gif(frames, OUT / "features.gif", seconds)
print(f"{path.name:<32} {len(frames)} frames, {path.stat().st_size / 1e6:.1f} MB")
print(took(start, "gifs"))


In [ ]:
# Pictures. The results chart from the compared rows of the scoring (stage_a_rows.csv, decoder rows of the four
# radar datasets, every model once) and the architecture picture.
start = time.perf_counter()
rows = pd.read_csv(RES / "stage_a_rows.csv")
rows = rows[(rows["head"] == "decoder") & rows["dataset"].isin(DATASETS)]
COLUMNS = ["dataset", "model", "mean", "sd"]
students = rows[["dataset", "model", "model_score", "model_sd"]].set_axis(COLUMNS, axis=1)
references = rows[["dataset", "reference", "reference_score", "reference_sd"]].set_axis(COLUMNS, axis=1)
table = pd.concat([students, references]).drop_duplicates(["dataset", "model"])
NAMES = {"student-f": "Student F", "student-n": "Student N", "student-c": "Student C (no radar teacher)",
         "dinov3-l-sat": "Frozen DINOv3 SAT", "terrafm-b-s1": "TerraFM-B reading S1", "student-m": "Part three student M"}
table["model"] = table["model"].map(NAMES)
order = list(NAMES.values())
colours = dict(zip(order, ["#e8a33d", "#f2c879", "#c9b28a", "#3b6fb6", "#6aa3d8", "#b5b5b5"]))
print(figures.results_chart(table, OUT / "results.png", order, list(DATASETS), colours).name)
print(figures.architecture_picture(OUT / "architecture.png").name)
print(took(start, "pictures"))


In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
